In [ ]:
%run validation

In [ ]:
from delta.tables import DeltaTable

In [ ]:
def is_table_exist(table_name, lakehouse="Bronze_lakehouse"):
    return spark.catalog.tableExists(f"{lakehouse}.dbo.{table_name}")

In [ ]:
def read_table(table_name, lakehouse="Bronze_lakehouse"):
    validate_table_name(table_name)
    return spark.read.table(f"{lakehouse}.dbo.{table_name}")

In [ ]:
def save_table(df, table_name, lakehouse="Silver_lakehouse", mode="overwrite"):
    validate_dataframe(df)
    validate_table_name(table_name)
    validate_save_mode(mode)
    df.write.format("delta").mode(mode).saveAsTable(f"{lakehouse}.dbo.{table_name}")

In [ ]:
def create_temp_view(df, view_name):
    df.createOrReplaceTempView(view_name)

In [ ]:
def create_table(df, table_name):
    df.write.format("delta").saveAsTable(table_name)

In [ ]:
def insert_new_rows(df, table_name, key_columns):
    condition = " AND ".join([f"target.{cl} = source.{cl}" for cl in key_columns])
    DeltaTable.forName(spark, table_name).alias("target")\
        .merge(df.alias("source"), condition)\
        .whenNotMatchedInsertAll().execute()

In [ ]:
def merge_table(df, table_name, key_columns, lakehouse="Silver_lakehouse"):
    validate_dataframe(df)
    validate_table_name(table_name)
    table = f"{lakehouse}.dbo.{table_name}"
    if not spark.catalog.tableExists(table):
        create_table(df, table)
    else:
        insert_new_rows(df, table, key_columns)